# Building a Chatbot with Memory

Build stateful chatbots that remember conversation history, manage context limits, and persist across sessions.

**Topics:** Conversation history, context window management, sliding window memory, summary memory, persistent storage

## 1. Basic Conversation Loop with History

In [ ]:
import os
from openai import OpenAI
from typing import TypedDict

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

class Message(TypedDict):
    role: str   # 'system' | 'user' | 'assistant'
    content: str

class Chatbot:
    """Simple stateful chatbot with full history."""
    
    def __init__(self, system_prompt: str, model: str = 'gpt-4o-mini'):
        self.model = model
        self.history: list[Message] = []
        self.system = system_prompt
    
    def chat(self, user_message: str) -> str:
        self.history.append({'role': 'user', 'content': user_message})
        
        messages = [{'role': 'system', 'content': self.system}] + self.history
        
        response = client.chat.completions.create(
            model=self.model,
            messages=messages,
        )
        reply = response.choices[0].message.content
        self.history.append({'role': 'assistant', 'content': reply})
        return reply
    
    def reset(self):
        self.history = []
    
    @property
    def turn_count(self) -> int:
        return len(self.history) // 2

# Demo conversation flow
bot = Chatbot(system_prompt='You are a helpful coding assistant.')

# Simulated conversation
conversation = [
    ('User', 'My name is Alex. I am learning Python.'),
    ('Bot', 'Nice to meet you, Alex! Python is a great choice...'),
    ('User', 'What is my name?'),  # Tests memory
    ('Bot', 'Your name is Alex, as you mentioned earlier.'),  # Bot remembers!
]

print('Simulated conversation with memory:')
for speaker, msg in conversation:
    print(f'  {speaker}: {msg}')
print()
print(f'History grows each turn: {len(conversation)//2} turns = {len(conversation)} messages')

## 2. Context Window Management — Truncation Strategies

In [ ]:
import tiktoken

def count_message_tokens(messages: list[dict], model: str = 'gpt-4o-mini') -> int:
    """Count tokens across all messages including overhead."""
    enc = tiktoken.encoding_for_model(model)
    total = 0
    for msg in messages:
        total += 4  # message overhead
        total += len(enc.encode(msg.get('content', '')))
    total += 2  # reply priming
    return total

class SlidingWindowChatbot:
    """Keeps the most recent N messages to stay within context limit."""
    
    def __init__(
        self,
        system_prompt: str,
        max_tokens: int = 4096,
        model: str = 'gpt-4o-mini',
    ):
        self.system = system_prompt
        self.max_tokens = max_tokens
        self.model = model
        self.history: list[Message] = []
    
    def _trim_history(self) -> list[Message]:
        """Remove oldest messages (in pairs) until under token limit."""
        history = list(self.history)
        system_msgs = [{'role': 'system', 'content': self.system}]
        
        while history:
            total = count_message_tokens(system_msgs + history, self.model)
            if total <= self.max_tokens:
                break
            # Remove oldest user+assistant pair
            history = history[2:]
        
        return history
    
    def chat(self, user_message: str) -> str:
        self.history.append({'role': 'user', 'content': user_message})
        trimmed = self._trim_history()
        
        messages = [{'role': 'system', 'content': self.system}] + trimmed
        response = client.chat.completions.create(model=self.model, messages=messages)
        reply = response.choices[0].message.content
        self.history.append({'role': 'assistant', 'content': reply})
        return reply

# Show token growth over conversation
print('Token growth simulation (100 tokens/turn):')
print(f'{"Turn":<6} {"Messages":<10} {"Tokens":<10} {"Action"}')
print('-' * 45)
for turn in range(1, 11):
    tokens = turn * 200  # ~100 user + 100 assistant
    action = 'TRIM' if tokens > 800 else 'OK'
    print(f'{turn:<6} {turn*2:<10} {tokens:<10} {action}')

## 3. Summary Memory — Compress Old Turns with LLM

In [ ]:
SUMMARIZE_PROMPT = """Summarize the following conversation into 3-5 bullet points.
Preserve key facts: names, decisions, preferences, and any important context.
This summary will be used as memory for future conversation turns.

Conversation:
{conversation}

Summary (bullet points):"""

class SummaryMemoryChatbot:
    """Summarizes old conversation to fit new turns in context."""
    
    def __init__(self, system_prompt: str, summarize_after: int = 10):
        self.system = system_prompt
        self.summarize_after = summarize_after  # summarize every N turns
        self.history: list[Message] = []
        self.summary: str = ''  # compressed memory of past turns
    
    def _should_summarize(self) -> bool:
        return len(self.history) >= self.summarize_after * 2
    
    def _summarize(self):
        """Compress old history into a summary, keep recent turns."""
        # Keep last 4 messages (2 turns) verbatim
        to_summarize = self.history[:-4]
        self.history = self.history[-4:]
        
        conv_text = '\n'.join(
            f"{m['role'].upper()}: {m['content']}" for m in to_summarize
        )
        
        # Use LLM to compress
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': SUMMARIZE_PROMPT.format(conversation=conv_text)}],
            max_tokens=300,
        )
        new_summary = response.choices[0].message.content
        
        # Append to existing summary
        if self.summary:
            self.summary += f'\n\nAdditional context:\n{new_summary}'
        else:
            self.summary = new_summary
    
    def _build_system(self) -> str:
        if self.summary:
            return f'{self.system}\n\n[Conversation history summary]:\n{self.summary}'
        return self.system

# Memory strategy comparison
strategies = [
    ('Full history', 'Keeps all messages', 'Best recall', 'Hits context limit'),
    ('Sliding window', 'Drops oldest messages', 'Constant tokens', 'Loses early context'),
    ('Summary memory', 'Compresses old turns', 'Long conversations', 'Lossy compression'),
    ('Hybrid', 'Summary + recent turns', 'Best balance', 'Extra LLM call cost'),
]
print(f'{"Strategy":<18} {"Method":<30} {"Pro":<22} {"Con"}')
print('-' * 85)
for name, method, pro, con in strategies:
    print(f'{name:<18} {method:<30} {pro:<22} {con}')

## 4. Persistent Chat History with JSON Storage

In [ ]:
import json
import uuid
from pathlib import Path
from datetime import datetime

class PersistentChatbot:
    """Chatbot with conversation persistence across sessions."""
    
    def __init__(
        self,
        system_prompt: str,
        storage_dir: str = './chat_sessions',
        session_id: str | None = None,
    ):
        self.system = system_prompt
        self.storage_dir = Path(storage_dir)
        self.storage_dir.mkdir(exist_ok=True)
        self.session_id = session_id or str(uuid.uuid4())[:8]
        self.history: list[Message] = self._load_history()
    
    def _session_path(self) -> Path:
        return self.storage_dir / f'{self.session_id}.json'
    
    def _load_history(self) -> list[Message]:
        path = self._session_path()
        if path.exists():
            data = json.loads(path.read_text())
            print(f'Loaded {len(data["messages"])} messages from session {self.session_id}')
            return data['messages']
        return []
    
    def _save_history(self):
        data = {
            'session_id': self.session_id,
            'created_at': datetime.now().isoformat(),
            'messages': self.history,
        }
        self._session_path().write_text(json.dumps(data, indent=2))
    
    def chat(self, user_message: str) -> str:
        self.history.append({'role': 'user', 'content': user_message})
        messages = [{'role': 'system', 'content': self.system}] + self.history
        response = client.chat.completions.create(model='gpt-4o-mini', messages=messages)
        reply = response.choices[0].message.content
        self.history.append({'role': 'assistant', 'content': reply})
        self._save_history()  # persist after every turn
        return reply

# Demo the session structure
sample_session = {
    'session_id': 'abc12345',
    'created_at': '2025-01-15T14:30:00',
    'messages': [
        {'role': 'user', 'content': 'Hello, I need help with Python.'},
        {'role': 'assistant', 'content': 'I would be happy to help with Python!'},
    ]
}
print('Session file structure:')
print(json.dumps(sample_session, indent=2))

## 5. Multi-Session Chatbot with User Isolation

In [ ]:
from collections import defaultdict
import hashlib

class MultiUserChatService:
    """Manages separate conversation histories for multiple users."""
    
    def __init__(self, system_prompt: str, max_history_per_user: int = 50):
        self.system = system_prompt
        self.max_history = max_history_per_user
        self._sessions: dict[str, list[Message]] = defaultdict(list)
    
    def _get_session_key(self, user_id: str, session_id: str) -> str:
        """Composite key: user_id + session_id for isolation."""
        return hashlib.sha256(f'{user_id}:{session_id}'.encode()).hexdigest()[:16]
    
    def chat(self, user_id: str, session_id: str, message: str) -> str:
        key = self._get_session_key(user_id, session_id)
        history = self._sessions[key]
        
        # Sliding window to limit memory per user
        if len(history) >= self.max_history:
            history = history[-self.max_history+2:]  # keep most recent
        
        history.append({'role': 'user', 'content': message})
        messages = [{'role': 'system', 'content': self.system}] + history
        
        response = client.chat.completions.create(model='gpt-4o-mini', messages=messages)
        reply = response.choices[0].message.content
        history.append({'role': 'assistant', 'content': reply})
        self._sessions[key] = history
        return reply
    
    def get_stats(self) -> dict:
        return {
            'active_sessions': len(self._sessions),
            'total_messages': sum(len(h) for h in self._sessions.values()),
        }

# Simulate multiple users
service = MultiUserChatService('You are a helpful assistant.')
print('Multi-user chat service initialized')
print('Key insight: Each (user_id, session_id) pair gets completely isolated history.')
print()
print('Session key examples:')
for user_id, session_id in [('user_001', 'sess_A'), ('user_001', 'sess_B'), ('user_002', 'sess_A')]:
    key = hashlib.sha256(f'{user_id}:{session_id}'.encode()).hexdigest()[:16]
    print(f'  {user_id} + {session_id} → {key}')
print()
print(service.get_stats())

## Exercises

1. **Conversation search:** Add a `search(query: str)` method to `PersistentChatbot` that finds the most relevant past messages using basic keyword matching or embeddings.

2. **Memory budget:** Implement a `BudgetChatbot` that tracks token usage per user, enforces a daily token limit, and raises a custom `BudgetExceededError` when exceeded.

3. **Context injection:** Build a chatbot that, at the start of each session, automatically retrieves the user's name and top 3 preferences from previous sessions and injects them into the system prompt.